# Factor ETFs — Live Test: the quant teardown 🏷️

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Beat SPY outright?: Busted](https://img.shields.io/badge/Beat_SPY_outright%3F-Busted-8b949e?style=flat-square)

**Claim under test:** USMV/MTUM/VLUE/QUAL promised academic factor exposure in a 0.15%/yr ETF wrapper — a decade-plus later, did each deliver (a) the exposure, (b) the alpha?

**Dedup guard:** [330-low-volatility-anomaly](../../330-low-volatility-anomaly/) and [242-quality-minus-junk](../../242-quality-minus-junk/) grade the *academic cross-sections*; the unit under test here is the **live product**, net of its own fee, since its own inception.

**Method skeleton.** Monthly total returns (yfinance auto-adjusted), sliced to the last complete month (2026-06-30); excess = minus prior-month-end ^IRX/12. Per fund: CAPM with Newey-West (lags 6; 3/12 in robustness) — alpha *t* and beta-vs-one *t*; paired moving-block bootstrap CI on the realized vol ratio; up/down capture; a **two-factor NW regression** on [market excess, realized style spread] for exposure delivery; a spread-sign month split (Welch *t* + 10,000-draw permutation placebo). The sector 12-1 WML proxy is formed on months *t−12…t−2* — known at the end of *t−1*, applied to month *t*: exactly ONE month of lag. No other trading rule exists in this study (the funds are buy-and-hold products).

Frozen headline run: [`docs/results.md`](../docs/results.md), as-of 2026-07-03, fingerprint `fd36c691c5cb`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"

from factor_etf_live_test import data, strategy as st

FUNDS = data.FUNDS
HAVE_REAL = data.have_real()
if HAVE_REAL:
    TAPE = data.load_tape()
    MRET = data.monthly_total_returns(TAPE[[c for c in TAPE.columns if c != "^IRX"]])
    RF = data.monthly_rf(TAPE["^IRX"])
    SPY = MRET["SPY"].dropna()
    WML = data.sector_momentum_spread(MRET[data.SECTORS].dropna())
    VMG = data.value_spread(MRET)
    QMB = data.quality_spread(MRET)
    SPREADS = {"MTUM": WML, "VLUE": VMG, "QUAL": QMB}

    def fund_frame(tk):
        r = MRET[tk].dropna()
        idx = r.index.intersection(SPY.index).intersection(RF.dropna().index)
        return r.loc[idx], SPY.loc[idx], RF.loc[idx]
else:
    TAPE = MRET = RF = SPY = WML = VMG = QMB = None
    SPREADS = {}
print("real cache present:", HAVE_REAL)

# frozen headline numbers (mirror of docs/results.md)
R = {'asof': '2026-07-03', 'last_month': '2026-06-30', 'fingerprint': 'fd36c691c5cb', 'funds': {'USMV': {'start': '2011-11', 'n': 176, 'beta': 0.689, 'se': 0.041, 't_b1': -7.52, 'r2': 74.0, 'alpha': 0.69, 't_a': 0.47, 'vr': 0.799, 'vr_lo': 0.741, 'vr_hi': 0.876, 'vol': 11.2, 'spy_vol': 14.0, 'up': 71.9, 'down': 65.9, 'cagr': 11.44, 'spy_cagr': 14.94, 'sh': 0.887, 'spy_sh': 0.959, 'dd': -19.1, 'w': 4.9, 'spy_w': 7.71, 'act': -28.9, 't_act': -1.94}, 'MTUM': {'start': '2013-05', 'n': 158, 'beta': 0.979, 'se': 0.051, 't_b1': -0.4, 'r2': 75.4, 'alpha': 2.65, 't_a': 1.1, 'vr': 1.129, 'vr_lo': 1.002, 'vr_hi': 1.223, 'vol': 16.3, 'spy_vol': 14.4, 'up': 99.9, 'down': 81.8, 'cagr': 16.8, 'spy_cagr': 14.38, 'sh': 0.934, 'spy_sh': 0.887, 'dd': -30.2, 'w': 7.72, 'spy_w': 5.86, 'act': 19.9, 't_act': 0.93}, 'VLUE': {'start': '2013-05', 'n': 158, 'beta': 1.077, 'se': 0.058, 't_b1': 1.34, 'r2': 76.6, 'alpha': -0.96, 't_a': -0.32, 'vr': 1.231, 'vr_lo': 1.073, 'vr_hi': 1.317, 'vol': 17.7, 'spy_vol': 14.4, 'up': 103.3, 'down': 106.9, 'cagr': 13.81, 'spy_cagr': 14.38, 'sh': 0.722, 'spy_sh': 0.887, 'dd': -29.0, 'w': 5.49, 'spy_w': 5.86, 'act': 0.2, 't_act': 0.01}, 'QUAL': {'start': '2013-08', 'n': 155, 'beta': 0.993, 'se': 0.021, 't_b1': -0.36, 'r2': 96.1, 'alpha': -0.23, 't_a': -0.28, 'vr': 1.013, 'vr_lo': 0.985, 'vr_hi': 1.055, 'vol': 14.7, 'spy_vol': 14.5, 'up': 98.2, 'down': 98.7, 'cagr': 13.75, 'spy_cagr': 14.14, 'sh': 0.834, 'spy_sh': 0.867, 'dd': -27.8, 'w': 5.28, 'spy_w': 5.52, 'act': -2.7, 't_act': -0.41}}, 'spy_dd': -23.9, 'style': {'MTUM': {'name': 'sector 12-1 WML', 'load': 0.326, 't': 7.45, 'r2': 81.9, 'pos': 96.2, 'neg': -64.5, 'diff': 160.7, 'welch': 4.59, 'p': 0.0}, 'VLUE': {'name': 'IWD-IWF value spread', 'load': 0.517, 't': 9.99, 'r2': 85.6, 'pos': 131.2, 'neg': -84.5, 'diff': 215.7, 'welch': 6.06, 'p': 0.0}, 'QUAL': {'name': 'SPHQ-SPY quality spread', 'load': 0.384, 't': 8.76, 'r2': 97.4, 'pos': 20.7, 'neg': -26.4, 'diff': 47.1, 'welch': 3.63, 'p': 0.0003}}, 'alpha_lags': {'USMV': (0.45, 0.47, 0.47), 'MTUM': (1.11, 1.1, 1.24), 'VLUE': (-0.36, -0.32, -0.31), 'QUAL': (-0.29, -0.28, -0.27)}, 'proxies': {'WML': (5.5, 0.28), 'VMG': (-4.4, -0.22), 'QMB': (-4.8, -0.42)}, 'syn': {'null': {'beta': 1.019, 't_b1': 1.82, 'load': 0.01, 't_l': 0.56, 'a': -0.56, 't_a': -1.02}, 'planted': {'beta': 0.717, 't_b1': -10.54, 'load': 0.51, 't_l': 27.37, 'a': 2.44, 't_a': 4.43}}}


real cache present: True


## 0 · Data stamp (cache-first, deterministic)

In [2]:
if HAVE_REAL:
    try:
        from quantlab import repro
        panel = pd.concat([MRET[['SPY'] + FUNDS], RF.rename('rf')], axis=1)
        print('fingerprint:', repro.fingerprint(panel), '(frozen:', R['fingerprint'] + ')')
    except Exception as e:
        print('quantlab.repro unavailable:', e)
    for tk in FUNDS:
        s = MRET[tk].dropna()
        print(f'{tk}: {s.index.min().date()} -> {s.index.max().date()}  ({len(s)} complete months)')
else:
    print('cache missing — the frozen numbers in R carry the notebook')


fingerprint: fd36c691c5cb (frozen: fd36c691c5cb)
USMV: 2011-11-30 -> 2026-06-30  (176 complete months)
MTUM: 2013-05-31 -> 2026-06-30  (158 complete months)
VLUE: 2013-05-31 -> 2026-06-30  (158 complete months)
QUAL: 2013-08-31 -> 2026-06-30  (155 complete months)


> 💡 **In plain words:** the fingerprint is a hash of the exact monthly panel behind the published verdict — if your rerun prints the same 12 characters, you are holding byte-for-byte the same data.

## 1 · Per-fund CAPM (excess-vs-excess, Newey-West)

The two headline statistics per fund: **alpha** (the premium the papers promised) and **beta vs 1** (the risk profile the label promised).

In [3]:
if HAVE_REAL:
    rows = []
    for tk in FUNDS:
        r, sp, f = fund_frame(tk)
        cp = st.capm(r - f, sp - f, lags=6)
        rows.append([tk, len(r), cp['beta'], cp['se_beta'], cp['t_beta_vs1'],
                     cp['alpha_ann']*100, cp['t_alpha'], cp['r2']*100])
    print(pd.DataFrame(rows, columns=['fund','n','beta','NW se','t(b vs 1)',
          'alpha %/yr','NW t(a)','R2 %']).round(3).to_string(index=False))
else:
    for tk, d in R['funds'].items():
        print(tk, 'beta', d['beta'], 't(b vs 1)', d['t_b1'], 'alpha', d['alpha'], 't', d['t_a'])


fund   n  beta  NW se  t(b vs 1)  alpha %/yr  NW t(a)   R2 %
USMV 176 0.689  0.041     -7.521       0.691    0.467 74.040
MTUM 158 0.979  0.051     -0.404       2.651    1.096 75.400
VLUE 158 1.077  0.058      1.335      -0.964   -0.320 76.587
QUAL 155 0.993  0.021     -0.357      -0.229   -0.279 96.062


USMV: beta **0.689**, *t*(β<1) = **-7.52** — the low-vol profile is delivered at overwhelming significance. MTUM/QUAL are ~market-beta (0.98/0.99); VLUE runs hot (1.08). Alphas: all |*t*| ≤ 1.24.

> 💡 **In plain words:** beta is 'how much of the market's movement the fund carries'. USMV carries 69% of it — exactly the min-vol pitch. Alpha is 'return you can't explain by carrying the market' — and nobody has any.

## 2 · USMV's mechanical promise — vol ratio with a paired block bootstrap

The vol reduction must show on the tape *with a CI* (sampling noise exists even for mechanical claims). Paired moving-block bootstrap (block 6), light draws in-notebook; canonical numbers from the 2,000-draw run in `results.md`.

In [4]:
if HAVE_REAL:
    for tk in FUNDS:
        r, sp, f = fund_frame(tk)
        vr = st.vol_ratio_ci(r, sp, n_draws=500, seed=601)
        cap = st.up_down_capture(r, sp)
        d = R['funds'][tk]
        print(f"{tk}: vol ratio {vr['obs']:.3f} (light CI [{vr['lo']:.3f}, {vr['hi']:.3f}]; "
              f"canonical [{d['vr_lo']:.3f}, {d['vr_hi']:.3f}])  "
              f"capture up {cap['up']*100:.1f}% / down {cap['down']*100:.1f}%")


USMV: vol ratio 0.799 (light CI [0.747, 0.880]; canonical [0.741, 0.876])  capture up 71.9% / down 65.9%


MTUM: vol ratio 1.129 (light CI [1.004, 1.225]; canonical [1.002, 1.223])  capture up 99.9% / down 81.8%


VLUE: vol ratio 1.231 (light CI [1.073, 1.310]; canonical [1.073, 1.317])  capture up 103.3% / down 106.9%


QUAL: vol ratio 1.013 (light CI [0.984, 1.055]; canonical [0.985, 1.055])  capture up 98.2% / down 98.7%


USMV's realized vol ratio is **0.799** (CI [0.741, 0.876]) — a **20% vol reduction** (CI 12–26%), the *low edge* of the promised 20–30%. Down-capture 65.9%. Note VLUE is **more** volatile than SPY (ratio 1.23) — 'value' in live form was a higher-beta, rougher ride.

## 3 · Exposure delivery — two-factor loadings + spread-sign splits

Fund excess on [SPY excess, realized style spread], NW lags 6. The spread loading is the exposure-delivery statistic. Splits: active return (fund − SPY) in factor-won vs factor-lost months, Welch *t*, permutation placebo (light 1,000 draws in-notebook; canonical 10,000-draw *p* from `results.md`).

In [5]:
if HAVE_REAL:
    for tk in ('MTUM', 'VLUE', 'QUAL'):
        r, sp, f = fund_frame(tk)
        sl = st.style_loading(r - f, sp - f, SPREADS[tk], lags=6)
        spl = st.spread_sign_split(r - sp, SPREADS[tk], n_draws=1000, seed=601)
        d = R['style'][tk]
        print(f"{tk}: loading {sl['loading']:+.3f}  NW t {sl['t_loading']:+.2f}  "
              f"R2 {sl['r2']*100:.1f}%   split diff {spl['diff_bps']:+.1f} bps "
              f"Welch t {spl['welch_t']:+.2f}  placebo p {spl['p_placebo']:.4f} "
              f"(canonical {d['p']:.4f})")


MTUM: loading +0.326  NW t +7.45  R2 81.9%   split diff +160.7 bps Welch t +4.59  placebo p 0.0000 (canonical 0.0000)


VLUE: loading +0.517  NW t +9.99  R2 85.6%   split diff +215.7 bps Welch t +6.06  placebo p 0.0000 (canonical 0.0000)

QUAL: loading +0.384  NW t +8.76  R2 97.4%   split diff +47.1 bps Welch t +3.63  placebo p 0.0000 (canonical 0.0003)


All three loadings clear the bar by a mile: MTUM **+0.326 (t 7.45)**, VLUE **+0.517 (t 9.99)**, QUAL **+0.384 (t 8.76)**; splits at Welch *t* 3.6–6.1, placebos ≤ 0.0003. Combined with USMV's *t*(β<1) = −7.52, **exposure delivery is REAL** on the real tape.

> 💡 **In plain words:** we checked whether each fund actually moves with the style on its label, using a factor we can compute ourselves with no look-ahead. All four do, unambiguously.

*Proxy caveats:* the quality spread uses SPHQ (independent provider), which switched to the S&P 500 Quality index in 2016; the WML proxy is sector-level (coarser than MSCI's stock-level momentum), which if anything *understates* MTUM's true loading.

## 4 · Alpha delivery — NW-lag robustness (the axis that fails)

In [6]:
if HAVE_REAL:
    for tk in FUNDS:
        r, sp, f = fund_frame(tk)
        ts = [st.capm(r - f, sp - f, lags=lg) for lg in (3, 6, 12)]
        print(f"{tk}: alpha {ts[1]['alpha_ann']*100:+.2f}%/yr   NW t = "
              + '  '.join(f"{t['t_alpha']:+.2f} (lags {lg})" for t, lg in zip(ts, (3,6,12))))
else:
    for tk, tt in R['alpha_lags'].items():
        print(tk, 'NW t at lags 3/6/12:', tt)


USMV: alpha +0.69%/yr   NW t = +0.45 (lags 3)  +0.47 (lags 6)  +0.47 (lags 12)
MTUM: alpha +2.65%/yr   NW t = +1.11 (lags 3)  +1.10 (lags 6)  +1.24 (lags 12)
VLUE: alpha -0.96%/yr   NW t = -0.36 (lags 3)  -0.32 (lags 6)  -0.31 (lags 12)
QUAL: alpha -0.23%/yr   NW t = -0.29 (lags 3)  -0.28 (lags 6)  -0.27 (lags 12)


No lag choice rescues anyone. MTUM peaks at *t* = 1.24 (lags 12). The literature bar — *REAL needs a robust t ≥ 2 on the real tape* — is not approached.

Why? The realized factors themselves paid nothing over the live window: sector WML **+5.5** bps/mo (NW *t* +0.28), IWD−IWF **-4.4** (*t* -0.22), SPHQ−SPY **-4.8** (*t* -0.42). Faithful trackers of flat factors produce zero alpha — McLean-Pontiff decay, live.

## 5 · Third axis — did ANY beat SPY outright?

In [7]:
if HAVE_REAL:
    for tk in FUNDS:
        r, sp, f = fund_frame(tk)
        sf, ss = st.ann_stats(r, f), st.ann_stats(sp, f)
        act = r - sp
        print(f"{tk}: CAGR {sf['cagr']*100:.2f}% vs SPY {ss['cagr']*100:.2f}%  "
              f"Sharpe {sf['sharpe']:.3f} vs {ss['sharpe']:.3f}  "
              f"active {act.mean()*1e4:+.1f} bps/mo  NW t {st.nw_tstat(act, lags=6):+.2f}")


USMV: CAGR 11.44% vs SPY 14.94%  Sharpe 0.887 vs 0.959  active -28.9 bps/mo  NW t -1.94
MTUM: CAGR 16.80% vs SPY 14.38%  Sharpe 0.934 vs 0.887  active +19.9 bps/mo  NW t +0.93
VLUE: CAGR 13.81% vs SPY 14.38%  Sharpe 0.722 vs 0.887  active +0.2 bps/mo  NW t +0.01
QUAL: CAGR 13.75% vs SPY 14.14%  Sharpe 0.834 vs 0.867  active -2.7 bps/mo  NW t -0.41


**Busted.** None beats SPY at significance; three of four lagged on CAGR. The sharpest number on the board is actually USMV's **−1.94** — the closest thing to a *significant* result on this axis is a factor fund significantly **losing** the outright race (while, to be fair, winning on its own risk-profile terms: Sharpe 0.887 vs 0.959 is a photo finish at 80% of the vol).

> 💡 **In plain words:** if what you wanted was 'more money than the S&P', none of these funds gave it to you, and the biggest one gave you visibly less. What USMV gave you instead is nearly the same reward-per-risk with smaller crashes — which is what its label, read carefully, actually promised.

## 6 · Synthetic control — planted-parameter recovery (machinery proof only)

Deterministic joint (market, spread, fund) world; the estimators must stay quiet on the null (β=1, loading=0, α=0) and recover planted parameters. Never cited in support of a stamp.

In [8]:
for label, kw in [('null    (b=1.0, s=0.0, a=+0%)', dict(beta=1.0, loading=0.0, alpha_ann=0.0)),
                  ('planted (b=0.7, s=0.5, a=+3%)', dict(beta=0.7, loading=0.5, alpha_ann=0.03))]:
    w = data.synthetic_world(n_months=168, seed=601, **kw)
    sl = st.style_loading(w['fund'], w['mkt'], w['spread'], lags=6)
    cp = st.capm(w['fund'], w['mkt'], lags=6)
    print(f"{label}: beta {cp['beta']:.3f} (t vs 1 {cp['t_beta_vs1']:+.2f})  "
          f"loading {sl['loading']:+.3f} (t {sl['t_loading']:+.2f})  "
          f"alpha {sl['alpha_ann']*100:+.2f}%/yr (t {sl['t_alpha']:+.2f})")


null    (b=1.0, s=0.0, a=+0%): beta 1.019 (t vs 1 +1.82)  loading +0.010 (t +0.56)  alpha -0.56%/yr (t -1.02)


planted (b=0.7, s=0.5, a=+3%): beta 0.717 (t vs 1 -10.54)  loading +0.510 (t +27.37)  alpha +2.44%/yr (t +4.43)


Null stays below the bar on all three estimators; the planted world is recovered (β 0.717, loading +0.510, α +2.44%/yr at *t* 4.43). The pipeline can detect exactly the effects it failed to find on the real tape — the nulls are informative.

## Verdict

- **Signal — REAL (exposure delivery).** USMV *t*(β<1) = −7.52 with a 20% vol reduction (CI 12–26%); style loadings *t* = +7.45 / +9.99 / +8.76; splits Welch *t* 3.6–6.1, placebo *p* ≤ 0.0003. All on the live tape, net of fees. Caveat: flagship-survivor selection (these are the four biggest surviving wrappers of the launch wave).
- **Tradability — FRAGILE.** Access is as good as it gets (0.15%/yr, penny spreads, huge AUM) — but the harvestable *premium* is absent: alphas −1.0% to +2.7%/yr, all |*t*| ≤ 1.24 at every lag; the style spreads themselves paid ~zero. You can cheaply buy a risk profile; you cannot buy the promised edge.
- **"Did any beat SPY outright?" — BUSTED.** None at significance; three of four lagged. MTUM's +2.42 pp/yr reads *t* = 0.93.

Frozen numbers: [`docs/results.md`](../docs/results.md) (as-of 2026-07-03, fingerprint `fd36c691c5cb`) · sources: [`docs/references.md`](../docs/references.md).

*Research & education, not investment advice.*